# 环节 13 · 开销实测与量级校准演示

纯 Python 标准库，零依赖。把实测数据变成可用的判断：

1. 开销占比模型（沙箱税 = 固定开销 × 创建次数 / 任务总时长）；
2. 固定开销**摊薄**：同一 5 ms 在不同任务长度下的含义；
3. 五个「不可比」陷阱的复现（同数据不同口径 → 不同结论）；
4. 容量规划：固定开销 × 并发。

> **本机实测（macOS Seatbelt / M4 Pro / 2026-09-12，中位数）**：
> `/bin/echo` 1.9 → 6.9 ms｜解释器初始化 21.5 → 30.4 ms｜
> 文件 IO 50.2 → 55.9 ms｜CPU 计算 82.4 → 84.2 ms｜回环 socket 22.6 → 27.5 ms
> → **固定开销 ≈ 5 ms/进程；CPU 型负载仅 +2%~3%**。

In [ ]:
# §1 开销占比模型（用本机实测的固定开销）
FIXED_MS = 5.0     # 本机实测：macOS Seatbelt 每进程固定开销 ≈ 5 ms


def sandbox_tax(task_ms, creations):
    overhead = FIXED_MS * creations
    total = task_ms + overhead
    return overhead, total, overhead / total


CASES = [
    ("单次短命令（每次新建）", 2, 1),
    ("跑一次单测（每次新建）", 2000, 1),
    ("会话复用 10 条命令", 30000, 1),
    ("批量 100 个小任务（每次新建）", 200, 100),
]
print(f"{'任务形态':<28}{'沙箱开销(ms)':<14}{'总时长(ms)':<14}{'沙箱税'}")
print("-" * 68)
for label, task_ms, n in CASES:
    oh, total, ratio = sandbox_tax(task_ms, n)
    print(f"{label:<28}{oh:<14.1f}{total:<14.1f}{ratio:.2%}")

print()
print("同一个 5 ms：短命令占 71%，长任务占 0.02% ——『沙箱免费/很贵』都对，取决于任务长度")

In [ ]:
# §2 固定开销摊薄：任务越长，比例越低
FIXED = 5.0
print(f"{'纯执行(ms)':<12}{'沙箱开销(ms)':<14}{'沙箱税':<10}{'判断'}")
print("-" * 52)
for task in [2, 20, 200, 2000, 30000]:
    total = task + FIXED
    ratio = FIXED / total
    if ratio > 0.5:
        verdict = "固定开销主导"
    elif ratio > 0.05:
        verdict = "需要考虑复用"
    else:
        verdict = "可忽略"
    print(f"{task:<12}{FIXED:<14.1f}{ratio:<10.2%}{verdict}")

print()
print("工程结论：①短任务高频 → 必须会话复用/预热池")
print("          ②长任务 → 不必为性能牺牲隔离档")

In [ ]:
# §3 五个「不可比」陷阱的复现：同一组数据，三种口径三个结论
MEASURED = [("短命令", 2.0, 7.0), ("文件IO", 50.0, 55.0), ("CPU计算", 82.0, 84.0)]

# 口径 A：只报「平均绝对开销」
abs_oh = [s - b for _, b, s in MEASURED]
print(f"口径A 平均绝对开销 = {sum(abs_oh) / len(abs_oh):.1f} ms  →『开销很小』")

# 口径 B：只报「平均相对开销」
rel = [(s - b) / b for _, b, s in MEASURED]
print(f"口径B 平均相对开销 = {sum(rel) / len(rel):.0%}      →『开销很大』")

# 口径 C：按任务长度分层（正确做法）
print("口径C 按任务长度分层：")
for name, b, s in MEASURED:
    print(f"    {name:<8} {b:>6.1f} → {s:>6.1f} ms   (+{(s - b) / b:.0%})")
print()
print("教训：口径 A/B 都能『自圆其说』却互相矛盾 —— 必须分层报数")
print("（另四个陷阱：冷启动 vs 稳态 / 粒度不对齐 / 配置不一致 / 单次采样）")

In [ ]:
# §4 容量规划：固定开销 × 并发
CORES = 12
FIXED_MS = 5.0


def max_rate(cores=CORES, fixed_ms=FIXED_MS, budget_ratio=0.2):
    """只用 budget_ratio 的 CPU 预算来"开沙箱"，能支撑多少 沙箱/秒"""
    budget_ms_per_s = cores * 1000 * budget_ratio
    return budget_ms_per_s / fixed_ms


r = max_rate()
print(f"12 核，允许 20% CPU 用于『开沙箱』，固定开销 {FIXED_MS} ms：")
print(f"  上限 ≈ {r:.0f} 个沙箱/秒")
print()
for c in [1, 4, 12, 48]:
    print(f"  {c:>3} 核 → {max_rate(cores=c):>6.0f} 个沙箱/秒")
print()
print("对比：若会话复用（一个沙箱跑 10 条命令），等效创建次数降到 1/10")
print("→ 同样的 CPU 预算可支撑 10 倍任务量 —— 这就是预热池/会话复用的量化价值")
print()
print("注意：容器/微 VM 的固定开销高 2~3 个数量级（引用量级 10²ms），")
print("      本表只适用于本机实测的进程级档位")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 沙箱开销是「算力损耗」吗？ | 不是；主要是**每进程固定开销**（本机实测 ≈ 5 ms） |
| 2 | 「沙箱几乎免费」对吗？ | 分任务长度：长任务 +2~3%，短命令可涨 2.6 倍 |
| 3 | profile 越严格越慢吗？ | 不是；实测 allow/strict 无显著差异 |
| 4 | 沙箱拦网络和网络不通怎么区分？ | 错误类型：`PermissionError`（策略）vs `ConnectionRefusedError`（网络） |
| 5 | 为什么白名单要数据驱动？ | 手写易漏；实测收紧后 `/bin/ls` 直接 SIGABRT |
| 6 | 容量规划为什么要乘并发？ | 单次 5 ms 很小，1000 次/秒就是 5 秒 CPU 时间 |

**相关长文**：[环节13-开销实测与量级校准详解.md](./环节13-开销实测与量级校准详解.md)